In [12]:
import pandas as pd

books = pd.read_csv("trimmed_books.csv")

In [3]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "cpu")
classifier("I love this!")

Device set to use cpu


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528691716492176},
  {'label': 'neutral', 'score': 0.005764589179307222},
  {'label': 'anger', 'score': 0.004419791977852583},
  {'label': 'sadness', 'score': 0.002092393347993493},
  {'label': 'disgust', 'score': 0.001611992483958602},
  {'label': 'fear', 'score': 0.0004138525982853025}]]

In [4]:
classifier(books["description"][0])

[[{'label': 'neutral', 'score': 0.5653485059738159},
  {'label': 'joy', 'score': 0.16986116766929626},
  {'label': 'surprise', 'score': 0.10455549508333206},
  {'label': 'fear', 'score': 0.08987922221422195},
  {'label': 'disgust', 'score': 0.04939651116728783},
  {'label': 'anger', 'score': 0.01208510436117649},
  {'label': 'sadness', 'score': 0.008874023333191872}]]

In [5]:
classifier(books["description"][0].split("."))

[[{'label': 'neutral', 'score': 0.5964019894599915},
  {'label': 'surprise', 'score': 0.20793794095516205},
  {'label': 'joy', 'score': 0.13256336748600006},
  {'label': 'fear', 'score': 0.029206110164523125},
  {'label': 'anger', 'score': 0.012968327850103378},
  {'label': 'disgust', 'score': 0.011465049348771572},
  {'label': 'sadness', 'score': 0.009457229636609554}],
 [{'label': 'neutral', 'score': 0.8186129927635193},
  {'label': 'joy', 'score': 0.061316363513469696},
  {'label': 'disgust', 'score': 0.04562665522098541},
  {'label': 'anger', 'score': 0.029492763802409172},
  {'label': 'fear', 'score': 0.017589688301086426},
  {'label': 'surprise', 'score': 0.01572026126086712},
  {'label': 'sadness', 'score': 0.011641278862953186}],
 [{'label': 'neutral', 'score': 0.6645634174346924},
  {'label': 'surprise', 'score': 0.14718058705329895},
  {'label': 'joy', 'score': 0.0702848881483078},
  {'label': 'disgust', 'score': 0.05568712577223778},
  {'label': 'sadness', 'score': 0.0294432

In [6]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
uvs = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [7]:
for i in range(10):
    uvs.append(books["unique_values"][i])
    sentences = books["description"][i]
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [8]:
emotion_scores

{'anger': [np.float64(0.01208510436117649),
  np.float64(0.0060784295201301575),
  np.float64(0.8568910956382751),
  np.float64(0.007311983034014702),
  np.float64(0.013028065674006939),
  np.float64(0.061268966645002365),
  np.float64(0.009007582440972328),
  np.float64(0.009723047725856304),
  np.float64(0.007103652693331242),
  np.float64(0.005231593269854784)],
 'disgust': [np.float64(0.04939651116728783),
  np.float64(0.007412190083414316),
  np.float64(0.0832119956612587),
  np.float64(0.01017354428768158),
  np.float64(0.04458419978618622),
  np.float64(0.39603734016418457),
  np.float64(0.004705113358795643),
  np.float64(0.004826457239687443),
  np.float64(0.014583729207515717),
  np.float64(0.018415164202451706)],
 'fear': [np.float64(0.08987922221422195),
  np.float64(0.0014112589415162802),
  np.float64(0.015202212147414684),
  np.float64(0.0062963152304291725),
  np.float64(0.011537071317434311),
  np.float64(0.012590764090418816),
  np.float64(0.006037553772330284),
  np.

In [11]:
# from tqdm import tqdm

# emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
# uvs = []
# emotion_scores = {label: [] for label in emotion_labels}

# for i in tqdm(range(len(books))):
#     uvs.append(books["unique_values"][i])
#     sentences = books["description"][i].split(".")
#     predictions = classifier(sentences)
#     max_scores = calculate_max_emotion_scores(predictions)
#     for label in emotion_labels:
#         emotion_scores[label].append(max_scores[label])

  4%|██▊                                                                       | 4943/128108 [10:49<4:29:34,  7.61it/s]


RuntimeError: The expanded size of the tensor (740) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 740].  Tensor sizes: [1, 514]

In [13]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
uvs = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    try:
        uvs.append(books["unique_values"][i])
        sentences = books["description"][i].split(".")
        predictions = classifier(sentences)
        max_scores = calculate_max_emotion_scores(predictions)

        for label in emotion_labels:
            emotion_scores[label].append(max_scores[label])

    except Exception as e:
        print(f"Error at index {i}: {e}")
        for label in emotion_labels:
            emotion_scores[label].append(None)

        continue

  4%|██▊                                                                       | 4230/109788 [09:24<4:03:17,  7.23it/s]

Error at index 4228: The expanded size of the tensor (740) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 740].  Tensor sizes: [1, 514]


  5%|███▊                                                                      | 5676/109788 [12:32<2:30:59, 11.49it/s]

Error at index 5675: The expanded size of the tensor (645) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 645].  Tensor sizes: [1, 514]


  5%|███▉                                                                     | 5896/109788 [13:02<25:43:44,  1.12it/s]

...

 50%|███████████████████████████████████▏                                   | 54392/109788 [1:59:07<1:27:28, 10.55it/s]

Error at index 54390: The expanded size of the tensor (711) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 711].  Tensor sizes: [1, 514]


 96%|████████████████████████████████████████████████████████████████████▉   | 105071/109788 [3:51:54<10:28,  7.50it/s]

Error at index 105068: The expanded size of the tensor (531) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 531].  Tensor sizes: [1, 514]


100%|████████████████████████████████████████████████████████████████████████| 109788/109788 [4:01:54<00:00,  7.56it/s]


In [15]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["unique_values"] = uvs

In [16]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,unique_values
0,0.034711,0.806784,0.082411,0.245415,0.825417,0.029443,0.264740,1
1,0.064134,0.104007,0.051363,0.040564,0.964106,0.111690,0.078765,2
2,0.974435,0.913045,0.351812,0.212945,0.549477,0.727175,0.185956,3
3,0.064134,0.104007,0.051363,0.040564,0.934228,0.111690,0.078765,4
4,0.053465,0.791916,0.059444,0.346165,0.878395,0.055465,0.156406,5
...,...,...,...,...,...,...,...,...
109783,0.064134,0.901087,0.959156,0.849096,0.785265,0.455001,0.340976,128104
109784,0.064134,0.104007,0.051363,0.887439,0.938217,0.111690,0.078765,128105
109785,0.064134,0.104007,0.051363,0.188964,0.967443,0.111690,0.078765,128106
109786,0.064134,0.104007,0.065212,0.248225,0.577747,0.111690,0.078765,128107


In [17]:
books = pd.merge(books, emotions_df, on = "unique_values")

In [18]:
books.to_csv("books_with_emotions.csv", index = False)